# M3.8 · Perceptron, TLU ve evrensel yaklaşım — derinlik, genişlik ve çift iniş

Biyolojik nörondan McCulloch–Pitts mantık birimlerine, eşik mantık birimi (TLU) ve Rosenblatt perceptronuna: öğrenme kuralı, yakınsama teoremi (Novikoff sınırı), XOR'un doğrusal ayrılamazlığı ve gizli katmanla çözümü. Doğrusal sınıflandırıcıya akıllı bir öznitelik dönüşümüyle (2B → 3B) ayrılabilirlik kazandırma, MLP'nin son katmanının da bir doğrusal sınıflandırıcı olduğu görüşü (önceki katmanlar dönüşümü öğrenir). Evrensel yaklaşım: ReLU birimleriyle parçalı doğrusal inşa; ders alıştırmasındaki gibi skaler, 2B Gauss ve vektör değerli fonksiyonları artan derinlikteki MLP'lerle yaklaşıklama; rastgele özniteliklerle en küçük normlu regresyonda çift iniş (double descent). Saf Python'da TLU/perceptron, NumPy'da rastgele öznitelik regresyonu ve çift iniş, PyTorch'ta derinlik deneyi ve öğrenilmiş gösterimlerin doğrusal ayrılabilirliği, sonda onlarca MLP'yi tek seferde eğiten toplu eğitim.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-8-perceptron.html](https://m-gonul.github.io/pytorch-lab/m3-8-perceptron.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_8_perceptron.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def logic():\n        t = need(ns, \"tlu\")\n        ensure([t((1, 1), -1.5, x) for x in ((0, 0), (0, 1), (1, 0), (1, 1))] == [0, 0, 0, 1], \"VE\")\n        ensure([t((1, 1), -0.5, x) for x in ((0, 0), (0, 1), (1, 0), (1, 1))] == [0, 1, 1, 1], \"VEYA\")\n        ensure([need(ns, \"xor_net\")(x) for x in ((0, 0), (0, 1), (1, 0), (1, 1))] == [0, 1, 1, 0], \"XOR ağı\")\n        X = [(0, 0), (0, 1), (1, 0), (1, 1)]\n        *_, conv = need(ns, \"train_perceptron\")(X, [0, 1, 1, 0], 1.0, 50)\n        ensure(not conv, \"XOR'da perceptron yakınsamamalı\")\n    R_.test(\"mantık birimleri ve XOR\", logic)\n\n    def conv():\n        rng = random.Random(0)\n        rows = []\n        for gamma in (0.4, 0.2, 0.1, 0.05):\n            X, y = [], []\n            while len(X) < 200:\n                p = (rng.uniform(-1, 1), rng.uniform(-1, 1))\n                s = (p[0] + 2 * p[1] - 0.3) / math.sqrt(5)\n                if abs(s) >= gamma:\n                    X.append(p)\n                    y.append(int(s > 0))\n            w, b, err, ok = ns[\"train_perceptron\"](X, y, 1.0, 1000)\n            ensure(ok and all(ns[\"tlu\"](w, b, x) == t for x, t in zip(X, y)), f\"γ = {gamma}: ayrılabilir veride yakınsamalı\")\n            R2 = max(1 + x[0] ** 2 + x[1] ** 2 for x in X)\n            bound = R2 / gamma ** 2\n            ensure(err <= bound, \"Novikoff sınırı (bias'ı ek bir girdi 1 olarak say)\")\n            rows.append([gamma, err, f\"{bound:.0f}\", f\"{need(ns, 'margin')(w, b, X, y):.3f}\"])\n        R_.table(\"Doğrusal ayrılabilir 200 örnek, gerçek kenar payı γ\", [\"γ\", \"perceptron hata sayısı\", \"Novikoff (R/γ)²\", \"bulunan çözümün kenar payı\"], rows,\n                 note=\"Sınır kötümserdir ama doğru ölçeklenir: γ yarıya inince hata sayısı kabaca 4 katına çıkar. Perceptron herhangi bir ayırıcı bulur, en geniş kenarlıyı değil (SVM'nin farkı, x1).\")\n    R_.test(\"yakınsama\", conv)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"relu_features\"), max_loops(src, 0, func=\"relu_interpolant\")))\n\n    def interp():\n        rng = np.random.default_rng(0)\n        k = np.sort(rng.uniform(-1, 1, 9))\n        v = rng.normal(size=9)\n        a, c0, c1 = need(ns, \"relu_interpolant\")(k, v)\n        ensure(len(a) == 7, \"k − 2 iç kırılma → 7 ReLU (uçlardaki iki düğüm doğrusal terimde)\")\n        x = np.linspace(k[0], k[-1], 400)\n        f = c0 + c1 * x + (a[None, :] * np.maximum(0, x[:, None] - k[None, 1:-1])).sum(1)\n        close(f, np.interp(x, k, v), atol=1e-9, what=\"parçalı doğrusal fonksiyon = ReLU toplamı\")\n    R_.test(\"ReLU ile parçalı doğrusal inşa\", interp)\n\n    def dd():\n        F = np.random.default_rng(1).normal(size=(5, 12))\n        y = np.arange(5.0)\n        w = need(ns, \"min_norm_fit\")(F, y)\n        close(F @ w, y, atol=1e-9, what=\"enterpolasyon\")\n        close(w, F.T @ np.linalg.solve(F @ F.T, y), atol=1e-9, what=\"en küçük norm = Fᵀ(FFᵀ)⁻¹y\")\n        res = need(ns, \"double_descent\")(trials=10)\n        ps = sorted(res)\n        peak = max(res, key=res.get)\n        ensure(90 <= peak <= 110, f\"tepe p ≈ n = 100'de olmalı (bulunan {peak})\")\n        ensure(res[3000] < res[100] / 10 and res[3000] < res[75], \"p ≫ n'de hata yeniden düşer\")\n        R_.table(\"Rastgele ReLU öznitelikleri, n = 100, d = 10, en küçük normlu çözüm (10 deneme ortancası)\", [\"p (öznitelik)\", \"test MSE\"], [[p, f\"{res[p]:.3f}\"] for p in ps],\n                 note=\"Klasik bölge (p < n): önce iyileşme, sonra aşırı uyum. Enterpolasyon eşiği (p = n): tek bir çözüm, gürültüye tam uyum, devasa norm. Aşırı parametreli bölge (p > n): sonsuz çözüm, en küçük normlu olan pürüzsüz — derin ağların neden aşırı parametreleyince bozulmadığının en basit modeli.\")\n    R_.test(\"çift iniş\", dd)\n",
 "d": "import math\nimport torch\nimport torch.nn as nn\nfrom labcheck import close, need, ensure\n\n\ndef rings(n, seed):\n    g = torch.Generator().manual_seed(seed)\n    r = torch.where(torch.rand(n, generator=g) < 0.5, 0.4, 1.0)\n    th = torch.rand(n, generator=g) * 2 * math.pi\n    X = torch.stack([r * torch.cos(th), r * torch.sin(th)], 1) + 0.08 * torch.randn(n, 2, generator=g)\n    return X, (r > 0.5).long()\n\n\ndef check(ns, R_):\n    def data():\n        for fn, di, do in ((\"f1\", 1, 1), (\"f2\", 2, 1), (\"f3\", 3, 2)):\n            X, Y = need(ns, \"make_data\")(fn, 50, 0)\n            ensure(X.shape == (50, di) and Y.shape == (50, do), f\"{fn} şekilleri\")\n        m = need(ns, \"mlp\")(3, 2, 3, 16)\n        ensure(sum(isinstance(l, nn.Linear) for l in m) == 4, \"3 gizli + çıkış\")\n    R_.test(\"veri ve mimari\", data)\n\n    def depth():\n        res = need(ns, \"depth_table\")([\"f1\", \"f2\", \"f3\"], [1, 2, 3, 4])\n        rows = [[fn] + [f\"{res[(fn, d)]:.2e}\" for d in (1, 2, 3, 4)] for fn in (\"f1\", \"f2\", \"f3\")]\n        for fn in (\"f1\", \"f2\", \"f3\"):\n            ensure(min(res[(fn, d)] for d in (2, 3, 4)) < res[(fn, 1)], f\"{fn}: derinlik yardımcı olmalı\")\n        R_.table(\"Test MSE: gizli katman sayısı (genişlik 64, Adam 2000 adım)\", [\"fonksiyon\", \"1\", \"2\", \"3\", \"4\"], rows,\n                 note=\"Aynı genişlikte bir katman çoğu zaman ilk iyileşmeyi yakalar ama keskin yapılar (f2'nin dar Gauss tepesi, f1'in salınımları) için derinlik çok daha verimli. Sonuçlar tohum ve optimizasyona da bağlı: farklar e aşamasında birçok tohumla ölçülüyor.\")\n    R_.test(\"ders alıştırması 1: derinlik\", depth)\n\n    def sep():\n        X, y = rings(600, 0)\n        torch.manual_seed(0)\n        m = need(ns, \"mlp\")(2, 1, 3, 32)\n        opt = torch.optim.Adam(m.parameters(), lr=1e-2)\n        for _ in range(800):\n            loss = nn.functional.binary_cross_entropy_with_logits(m(X)[:, 0], y.float())\n            opt.zero_grad()\n            loss.backward()\n            opt.step()\n        acc_in = need(ns, \"penultimate_separability\")(m, X, y)\n        lin = nn.Linear(2, 1)\n        optl = torch.optim.Adam(lin.parameters(), lr=0.05)\n        for _ in range(500):\n            l = nn.functional.binary_cross_entropy_with_logits(lin(X)[:, 0], y.float())\n            optl.zero_grad()\n            l.backward()\n            optl.step()\n        acc_raw = float(((lin(X)[:, 0] > 0).long() == y).float().mean())\n        ensure(acc_in > 0.95 and acc_raw < 0.75, \"ham uzayda doğrusal ayrılamaz, son gizli katmanda ayrılabilir\")\n        R_.table(\"Halkalar (ders alıştırması 3)\", [\"girdi\", \"doğrusal sınıflandırıcı doğruluğu\"], [[\"ham (x₁, x₂)\", f\"{acc_raw:.2f}\"], [\"MLP'nin son gizli katmanı (32 boyut)\", f\"{acc_in:.2f}\"]],\n                 note=\"MLP'nin gizli katmanları, x1'deki elle tasarlanmış (x₁, x₂, x₁² + x₂²) dönüşümünün işini öğrenir.\")\n    R_.test(\"öğrenilmiş öznitelikler\", sep)\n",
 "e": "import torch\nfrom labcheck import need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        m = need(ns, \"BatchedMLP\")(5, 3, 8, 2, 2)\n        ensure(tuple(m(torch.randn(5, 7, 3)).shape) == (5, 7, 2), \"şekil\")\n        x = torch.randn(1, 4, 3).expand(5, 4, 3)\n        out = m(x)\n        ensure(not torch.allclose(out[0], out[1]), \"her model kendi ağırlıkları\")\n        rows = []\n        for fn in (\"f1\", \"f2\"):\n            r = {d: need(ns, \"train_batched\")(16, fn, d, 1200) for d in (1, 2, 3)}\n            for d, v in r.items():\n                q = torch.quantile(v, torch.tensor([0.1, 0.5, 0.9]))\n                rows.append([fn, d, f\"{float(q[1]):.2e}\", f\"[{float(q[0]):.1e}, {float(q[2]):.1e}]\"])\n            ensure(float(r[3].median()) < float(r[1].median()), f\"{fn}: ortanca hata derinlikle düşmeli\")\n        tb = bench(lambda: ns[\"train_batched\"](16, \"f2\", 2, 300), repeat=1, number=1)\n        tl = bench(lambda: need(ns, \"train_loop\")(4, \"f2\", 2, 300), repeat=1, number=1) * 4\n        R_.table(\"16 model × (veri tohumu), genişlik 64\", [\"fonksiyon\", \"derinlik\", \"ortanca test MSE\", \"%10–%90 aralığı\"], rows,\n                 note=f\"Süre (300 adım, 16 model): toplu {fmt_time(tb)}, döngü (tahmini) {fmt_time(tl)}. Tek bir tohumla yapılan karşılaştırma (d aşaması) yanıltıcı olabilir; dağılım raporla.\")\n    R_.test(\"toplu eğitim\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        X, y = need(ns, \"make_rings\")(400, 1)\n        ensure(X.shape == (400, 2), \"şekil\")\n        L = need(ns, \"lift\")(X)\n        ensure(L.shape == (400, 3), \"3B\")\n        (pa, pm), (sa, sm) = need(ns, \"fit_both\")(L, y)\n        ensure(pa == 1.0 and sa == 1.0, \"3B'de ikisi de ayırmalı\")\n        ensure(sm >= pm - 1e-9, \"SVC kenar payı en büyük olmalı\")\n        R_.table(\"Kaldırılmış halkalar (x₁, x₂, x₁² + x₂²)\", [\"model\", \"doğruluk\", \"geometrik kenar payı\"], [[\"perceptron\", f\"{pa:.2f}\", f\"{pm:.3f}\"], [\"doğrusal SVC (C = 1000)\", f\"{sa:.2f}\", f\"{sm:.3f}\"]],\n                 note=\"İkisi de 3B'de bir düzlem bulur; perceptron'unki ilk hatasız düzlemdir, SVC en geniş kenarlıyı arar — yeni verilere daha sağlam. 2B'ye geri izdüşürülen karar sınırı bir çemberdir: x₁² + x₂² = sabit.\")\n    R_.test(\"perceptron vs SVC\", t)\n",
 "x2": "import random\nfrom itertools import product\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = random.Random(0)\n        rows = []\n        for n in (2, 3, 4, 5):\n            for _ in range(5):\n                table = {row: rng.randint(0, 1) for row in product((0, 1), repeat=n)}\n                f = lambda *r: table[r]\n                net = need(ns, \"truth_table_net\")(f, n)\n                ensure(all(need(ns, \"evaluate\")(net, r) == table[r] for r in table), f\"n = {n}: rastgele fonksiyon tam temsil\")\n            parity = lambda *r: sum(r) % 2\n            rows.append([n, len(ns[\"truth_table_net\"](parity, n)[0]), 2 ** n])\n        R_.table(\"Eşlik (parity) fonksiyonu için DNF gizli birim sayısı\", [\"n\", \"gizli TLU\", \"2ⁿ\"], rows,\n                 note=\"İki katman her Boole fonksiyonunu temsil eder ama eşlik için 2ⁿ⁻¹ birim gerekir; derin bir ağ (XOR'ların ağacı) aynı işi ~n birimle yapar — derinliğin üstel verimliliği.\")\n    R_.test(\"DNF inşası\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        db, dW = need(ns, \"bias_vs_weight_grads\")([0.0, 1e-4, 0.9], [0.5, -0.2])\n        close(db, np.array([0.5, -0.2]), what=\"bias gradyanı = δ\")\n        close(dW, np.outer([0.5, -0.2], [0.0, 1e-4, 0.9]), what=\"ağırlık gradyanı = δ aᵀ\")\n        zs = need(ns, \"zigzag_fraction\")(100000, \"sigmoid\", 1)\n        zt = ns[\"zigzag_fraction\"](100000, \"tanh\", 1)\n        ensure(zs == 1.0 and abs(zt - 0.5) < 0.02, \"sigmoid girdilerinde tüm güncellemeler aynı işaretli\")\n        close(need(ns, \"derivative_max\")(\"sigmoid\"), 0.25, atol=1e-6, what=\"σ' ≤ 1/4\")\n        close(ns[\"derivative_max\"](\"tanh\"), 1.0, atol=1e-6, what=\"tanh' ≤ 1\")\n        R_.table(\"Ders alıştırması 2\", [\"soru\", \"cevap\"],\n                 [[\"bias ile ağırlık güncellemesinin farkı\", \"∂C/∂b = δ, ∂C/∂w = a^{l−1}·δ: ağırlık güncellemesi önceki aktivasyonla ölçeklenir\"],\n                  [\"önceki aktivasyon ≈ 0 ise\", \"o ağırlık 'yavaş öğrenir' (ölü/doygun nöronların bağlantıları)\"],\n                  [\"sigmoid eksikleri\", f\"sıfır merkezli değil (güncellemelerin %{100 * zs:.0f}'i aynı işaretli → zikzak), türevi en çok 0.25 (katman başına ≥4× sönüm), doygunluk\"],\n                  [\"tanh eksiği\", \"sıfır merkezli ama yine doyar: |z| büyükken türev ≈ 0\"]],\n                 note=\"Sigmoid çıktıları hep pozitif olduğundan bir nörona gelen tüm ağırlıkların gradyanları aynı işareti taşır: ağırlık vektörü yalnız 'hepsi artar' ya da 'hepsi azalır' yönünde gidebilir.\")\n    R_.test(\"aktivasyonlar ve geri yayılım\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_8_perceptron as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Biyolojik ilham:** nöron, dendritlerden gelen sinyalleri toplar, eşik aşılırsa akson boyunca ateşler. McCulloch & Pitts (1943): ikili girdi/çıktılı yapay nöronla VE, VEYA, DEĞİL hesaplanır; ağları her mantıksal önermeyi hesaplar.

**TLU** (eşik mantık birimi): $z=w^\top x+b$, $\hat y=\mathrm{step}(z)$ — doğrusal bir karar sınırı. **Perceptron** (Rosenblatt 1957): TLU'lardan oluşan tek katman; öğrenme kuralı her hatalı örnekte $w\leftarrow w+\eta(y-\hat y)x$ ("birlikte ateşleyen birlikte bağlanır"ın hataya dayalı biçimi). **Yakınsama teoremi:** veri bir γ kenar payıyla doğrusal ayrılabiliyorsa ve $\|x\|\le R$ ise hata sayısı $\le(R/\gamma)^2$. Ayrılabilir değilse dolaşır (XOR).

**Doğrusal sınıflandırıcı + dönüşüm:** 2B halkalar doğrusal ayrılamaz; $(x_1,x_2)\mapsto(x_1,x_2,x_1^2+x_2^2)$ ile 3B'de bir düzlem ayırır (çekirdek yönteminin fikri). **MLP** aynı nöron modelini yığar (ağırlıklı toplam + doğrusalsızlık, tekrar tekrar): son katman, gizli katmanların **öğrendiği** dönüşüm üzerinde bir doğrusal sınıflandırıcıdır.

**Evrensel yaklaşım** (Cybenko 1989, Hornik 1991): tek gizli katman, yeterli genişlikle kompakt bir kümedeki her sürekli fonksiyonu yaklaşıklar. 1B'de k kırılma noktalı parçalı doğrusal fonksiyon k ReLU ile tam ifade edilir. Ama genişlik üstel büyüyebilir; **derinlik** bileşke yoluyla parçaları üstel çoğaltır (testere dişi örneği). Ders alıştırması: `MLPRegressor` ile katman sayısını artırarak skaler (1B), 2B Gauss $e^{-(x_1^2+x_2^2)/0.1}$ ve iki çıktılı vektör fonksiyonu yaklaşıklamak.

**Çift iniş:** klasik görüş — kapasite arttıkça test hatası U çizer. Rastgele özniteliklerle en küçük normlu çözümde, parametre sayısı p veri sayısı n'ye yaklaşınca hata patlar (enterpolasyon eşiği), p ≫ n'de yeniden düşer ve çoğu zaman U'nun dibinin altına iner.

## 1 · Elle

<!-- çözüm -->

AND: $1+1-1.5=0.5\ge0$ → 1; $(1,0)$: −0.5 → 0. Perceptron: $z=0.5-1=-0.5$ → ŷ = 0, y − ŷ = 1 → $w=(1.5,0)$, $b=1$.

Novikoff: $(2/0.5)^2=16$ — boyuttan bağımsız! XOR = (x₁ VEYA x₂) VE DEĞİL(x₁ VE x₂): iki gizli TLU + bir çıkış. Parametre: $235\,500+30\,100+1\,010$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**TLU, adım fonksiyonu $\mathrm{step}(z)=\mathbb 1[z\ge0]$**

- `andg`: w = (1, 1), b = −1.5: girdi (1, 1) için çıktı
- `and0`: Aynı TLU, girdi (1, 0)

**Perceptron güncellemesi: w = (0.5, −1), b = 0, x = (1, 1), y = 1, η = 1**

- `w1`: Yeni $w_1$
- `bb`: Yeni b

**Kuram ve sayım**

- `nov`: Novikoff: R = 2, kenar payı γ = 0.5 → en çok hata sayısı
- `xor`: XOR için gereken en az gizli TLU
- `par`: 784-300-100-10 MLP'nin parametre sayısı

In [ ]:
cevaplar = dict(
    andg=None,
    and0=None,
    w1=None,
    bb=None,
    nov=None,
    xor=None,
    par=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"andg":[1,0],"and0":[0,0],"w1":[1.5,0],"bb":[1,0],"nov":[16,0],"xor":[2,0],"par":[266610,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `tlu(w, b, x)` → 0/1.
- `train_perceptron(X, y, eta=1.0, max_epochs=100)` → `(w, b, hata_sayısı, yakınsadı_mı)`: her örnekte kural; bir epoch hatasız geçerse dur. w, b sıfırdan başlar.
- `margin(w, b, X, y)` — geometrik kenar payı $\min_i y_i^\pm(w^\top x_i+b)/\|w\|$, $y^\pm=2y-1$.
- `xor_net(x)` — iki gizli TLU + çıkış TLU, ağırlıkları sen seç.

In [ ]:
%%writefile stage_b.py
def tlu(w, b, x):
    ...


def train_perceptron(X, y, eta=1.0, max_epochs=100):
    ...


def margin(w, b, X, y):
    ...


def xor_net(x):
    ...


print([xor_net(x) for x in ((0, 0), (0, 1), (1, 0), (1, 1))])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Rastgele öznitelik regresyonu ve çift iniş**, NumPy, döngü yalnız p ve deneme üzerinde.

- `relu_features(X, W)` — $\max(0, XW)$.
- `min_norm_fit(F, y)` — en küçük normlu en küçük kareler (`np.linalg.pinv` ya da `lstsq`).
- `double_descent(n=100, d=10, ps=…, trials=20, sigma=0.3, seed=0)` → `{p: ortanca test MSE}`: hedef $\tanh(2x^\top\beta)$ + gürültü, $W\sim N(0,1/d)$.
- `relu_interpolant(knots, values)` — 1B: verilen düğümlerden geçen parçalı doğrusal fonksiyonu **k ReLU** + doğrusal terimle temsil et → `(a, c0, c1)` öyle ki $f(x)=c_0+c_1x+\sum_ja_j\max(0,x-k_j)$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def relu_features(X, W):
    ...


def min_norm_fit(F, y):
    ...


def double_descent(n=100, d=10, ps=(10, 25, 50, 75, 90, 100, 110, 125, 150, 200, 400, 1000, 3000), trials=20, sigma=0.3, seed=0):
    ...


def relu_interpolant(knots, values):
    ...


print(relu_interpolant(np.array([0.0, 1, 2, 3]), np.array([0.0, 1, 0, 2])))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. **Ders alıştırması 1:** artan derinlikle fonksiyon yaklaşımı.

- `make_data(fn, n, seed)` — `fn`: `"f1"` skaler $\sin(3\pi x)/(1+x^2)$, $x\sim U[-1,1]$; `"f2"` $e^{-(x_1^2+x_2^2)/0.1}$, $x\sim U[-1,1]^2$; `"f3"` iki çıktılı $\big[e^{-(x_1^2+x_2^2-2x_3)/0.1},\;e^{-(x_1^2+x_2^2+2x_3)/0.1}\big]$, $x\sim U[-0.5,0.5]^3$ (alıştırmanın 2. ve 3. maddeleri).
- `mlp(d_in, d_out, depth, width=64)` — `depth` gizli katman (ReLU).
- `fit(model, X, Y, steps=2000, lr=3e-3, seed=0)` → eğitim sonrası MSE (Adam, tam batch).
- `depth_table(fns, depths, n=2000)` → `{(fn, depth): test MSE}`.
- `penultimate_separability(model, X, y)` — eğitilmiş bir sınıflandırıcının son gizli katman aktivasyonlarına doğrusal sınıflandırıcı (lojistik regresyon, PyTorch) sığdır; doğruluk.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn


def make_data(fn, n, seed):
    ...


def mlp(d_in, d_out, depth, width=64):
    ...


def fit(model, X, Y, steps=2000, lr=3e-3, seed=0):
    ...


def depth_table(fns, depths, n=2000):
    ...


def penultimate_separability(model, X, y):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Toplu eğitim:** E adet aynı mimarili MLP'yi tek optimizasyonda eğit — ağırlıklar `(E, d_in, h)` tensörlerinde, ileri geçiş `torch.baddbmm`. Her model kendi verisini (farklı tohum) görür.

- `BatchedMLP(E, d_in, h, depth, d_out)` — `forward(X)`, `X (E, n, d_in)` → `(E, n, d_out)`.
- `train_batched(E, fn, depth, steps=1500, n=1000)` → `(E,)` test MSE'leri.
- `train_loop(E, fn, depth, steps=1500, n=1000)` — aynısını E kez ayrı `nn.Sequential` ile (karşılaştırma için).

In [ ]:
%%writefile stage_e.py
import math
import torch
import torch.nn as nn


class BatchedMLP(nn.Module):
    def __init__(self, E, d_in, h, depth, d_out):
        super().__init__()
        ...

    def forward(self, X):
        ...


def train_batched(E, fn, depth, steps=1500, n=1000):
    ...


def train_loop(E, fn, depth, steps=1500, n=1000):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Dönüştürülmüş veride perceptron ve SVC karar sınırları

**Ders alıştırması 3:** `make_circles` benzeri veri (NumPy ile üret: iki eş merkezli halka + gürültü).

- `make_rings(n, seed)` → `(X, y)`; `lift(X)` → $(x_1,x_2,x_1^2+x_2^2)$.
- `fit_both(X3, y)` → perceptron (`sklearn.linear_model.Perceptron`) ve doğrusal SVC (`sklearn.svm.SVC(kernel="linear", C=1e3)`) için `(eğitim doğruluğu, geometrik kenar payı)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def make_rings(n, seed):
    ...


def lift(X):
    ...


def fit_both(X3, y):
    ...


X, y = make_rings(200, 0)
print(fit_both(lift(X), y))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Her Boole fonksiyonu iki katman TLU'dur (VE-VEYA inşası)

- `truth_table_net(f, n)` — n girdili Boole fonksiyonu `f` (0/1 döndüren Python fonksiyonu) için VE-VEYA (DNF) inşası: doğru satır başına bir gizli TLU (o satırı ve yalnız onu tanıyan), çıkış VEYA TLU'su. `(gizli_ağırlıklar, gizli_biaslar, çıkış_ağırlıkları, çıkış_biası)`.
- `evaluate(net, x)`.

In [ ]:
%%writefile extra_x2.py
from itertools import product


def truth_table_net(f, n):
    ...


def evaluate(net, x):
    ...


xor = lambda a, b: a ^ b
print([evaluate(truth_table_net(xor, 2), x) for x in product((0, 1), repeat=2)])

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Sigmoid ve tanh: sıfır merkezlilik, doygunluk ve küçük aktivasyonlar

**Ders alıştırması 2:** geri yayılım denklemleriyle düşün.

- `bias_vs_weight_grads(a_prev, delta)` — katman l için $\partial C/\partial b_j=\delta_j$, $\partial C/\partial w_{jk}=a^{l-1}_k\delta_j$: önceki aktivasyon ≈ 0 ise ağırlık öğrenmez, bias öğrenir.
- `zigzag_fraction(n_steps, act, seed)` — iki girdili bir nörona her zaman pozitif girdi (sigmoid çıktısı gibi) gelirse, ağırlık gradyanlarının işaretleri hep aynı → güncellemeler yalnız (+,+) ya da (−,−) kadranlarında: bu kadranlara düşen güncelleme oranı. `act ∈ {"sigmoid", "tanh"}` önceki katmanın aktivasyonu.
- `derivative_max(act)` — σ' en çok 0.25, tanh' en çok 1.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def bias_vs_weight_grads(a_prev, delta):
    ...


def zigzag_fraction(n_steps, act, seed=0):
    ...


def derivative_max(act):
    ...


print(derivative_max("sigmoid"), derivative_max("tanh"))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Belkin ve ark. (2019): Çift iniş

[Belkin, Hsu, Ma, Mandal — Reconciling modern ML practice and the classical bias–variance trade-off (PNAS 2019)](https://doi.org/10.1073/pnas.1903070116)

Makalenin Şekil 2 deneyini MNIST'in 10 000 örneklik alt kümesinde rastgele Fourier öznitelikleriyle (RFF) yeniden üret: öznitelik sayısı N = 10³ … 6·10⁴, en küçük normlu çözüm, sıfır-bir ve kare kayıp, ağırlık normu. Ardından c aşamasındaki deneye ridge düzenlemesi ekle ve λ arttıkça tepenin nasıl kaybolduğunu göster; Nakkiran ve ark. (2020)'deki "epoch-wise" çift inişi küçük bir CNN'de (CIFAR-10 %20 etiket gürültüsü) gözlemle.


**Adımlar**

1. Çözüm için `torch.linalg.lstsq` ya da çekirdek trick'i (N > n'de n × n sistem).

**Kabul ölçütleri**

- Test hatası N ≈ n'de tepe yapar, N ≫ n'de klasik U'nun en iyisinden düşer.
- Ağırlık normu da N ≈ n'de tepe yapar.

## Makale ödevi: Cybenko (1989): Sigmoidlerin süperpozisyonuyla yaklaşım

[Cybenko — Approximation by Superpositions of a Sigmoidal Function (MCSS 1989)](https://doi.org/10.1007/BF02551274)

Teoremin yapıcı bir sürümünü uygula: [0,1]'de sürekli f için dik sigmoidlerin farklarından "basamaklar" kurarak N birimli tek gizli katmanlı ağı elle oluştur (eğitim yok) ve sup hatasını N'ye karşı çiz. Aynı fonksiyonu derin ama dar (ör. 4 × 8) eğitilmiş bir ağla karşılaştır; Telgarsky (2016)'nın derinlik ayrımı örneği olan "testere dişi" (tent map'in k kez bileşkesi) fonksiyonunun sığ ağla üstel genişlik gerektirdiğini deneyle göster.


**Adımlar**

1. Testere dişi: k ReLU katmanı ile 2^k doğrusal parça; sığ ağın aynı hatayı yakalamak için gereken genişliği ölç.

**Kabul ölçütleri**

- Elle kurulan ağın hatası ~1/N azalır.
- Testere dişinde derin ağ birkaç birimle, sığ ağ ~2^k birimle başarılı.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 10](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Rosenblatt — The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain (Psych. Review 1958)](https://doi.org/10.1037/h0042519) — ©
- [Novikoff — On Convergence Proofs on Perceptrons (1962)](https://apps.dtic.mil/sti/citations/AD0298258) — kamu
- [Cybenko — Approximation by Superpositions of a Sigmoidal Function (MCSS 1989)](https://doi.org/10.1007/BF02551274) — ©
- [Belkin, Hsu, Ma, Mandal — Reconciling modern machine-learning practice and the classical bias–variance trade-off (PNAS 2019)](https://doi.org/10.1073/pnas.1903070116) — ©
- [Nakkiran ve ark. — Deep Double Descent (ICLR 2020)](https://arxiv.org/abs/1912.02292) — arXiv